In [1]:
import numpy as np
import polars as pl
from tem import world
config_file = "../envs/mckenzie2024.json"

In [2]:
df = world.generate_mckenzie(config_file)
index = np.random.choice([0, 1], size=df.shape[0])
df.with_columns(
    action=np.array(["left", "right"])[index],
    object=pl.concat_list("object1", "object2").list.get(index),
    valence=pl.concat_list("valence1", "valence2").list.get(index),
)

phase,trial,context,object_set,object1,object2,valence1,valence2,action,object,valence
str,i64,str,str,str,str,str,str,str,str,str
"""AB""",1,"""C1""","""AB""","""A""","""B""","""reward""","""no reward""","""right""","""B""","""no reward"""
"""AB""",2,"""C2""","""AB""","""A""","""B""","""no reward""","""reward""","""left""","""A""","""no reward"""
"""AB""",3,"""C1""","""AB""","""B""","""A""","""no reward""","""reward""","""left""","""B""","""no reward"""
"""AB""",4,"""C2""","""AB""","""A""","""B""","""no reward""","""reward""","""right""","""B""","""reward"""
"""AB""",5,"""C1""","""AB""","""A""","""B""","""reward""","""no reward""","""left""","""A""","""reward"""
…,…,…,…,…,…,…,…,…,…,…
"""ABCD""",296,"""C2""","""AB""","""A""","""B""","""no reward""","""reward""","""left""","""A""","""no reward"""
"""ABCD""",297,"""C1""","""AB""","""B""","""A""","""no reward""","""reward""","""left""","""B""","""no reward"""
"""ABCD""",298,"""C2""","""AB""","""A""","""B""","""no reward""","""reward""","""left""","""A""","""no reward"""


In [3]:
index = np.random.choice([0, 1], size=df.shape[0])
df_trial = df.select(
    "phase",
    "trial",
    "context",
    "object_set",
    situation=pl.concat_str("context", "object1", "object2"),
    action=np.array(["left", "right"])[index],
    object=pl.concat_list("object1", "object2").list.get(index),
    valence=pl.concat_list("valence1", "valence2").list.get(index),
)
df_trial

phase,trial,context,object_set,situation,action,object,valence
str,i64,str,str,str,str,str,str
"""AB""",1,"""C1""","""AB""","""C1AB""","""left""","""A""","""reward"""
"""AB""",2,"""C2""","""AB""","""C2AB""","""right""","""B""","""reward"""
"""AB""",3,"""C1""","""AB""","""C1BA""","""right""","""A""","""reward"""
"""AB""",4,"""C2""","""AB""","""C2AB""","""right""","""B""","""reward"""
"""AB""",5,"""C1""","""AB""","""C1AB""","""left""","""A""","""reward"""
…,…,…,…,…,…,…,…
"""ABCD""",296,"""C2""","""AB""","""C2AB""","""right""","""B""","""reward"""
"""ABCD""",297,"""C1""","""AB""","""C1BA""","""left""","""B""","""no reward"""
"""ABCD""",298,"""C2""","""AB""","""C2AB""","""right""","""B""","""reward"""


In [4]:
design = (
    pl.concat(
        [
            df_trial.with_columns(trial_type=pl.lit("choice")),
            df_trial.with_columns(trial_type=pl.lit("feedback")),
        ]
    ).sort("phase", "trial", "context", "trial_type")
    .with_columns(
        node=pl.when(pl.col("trial_type") == "choice").then("situation").otherwise("valence")
    )
)
design

phase,trial,context,object_set,situation,action,object,valence,trial_type,node
str,i64,str,str,str,str,str,str,str,str
"""AB""",1,"""C1""","""AB""","""C1AB""","""left""","""A""","""reward""","""choice""","""C1AB"""
"""AB""",1,"""C1""","""AB""","""C1AB""","""left""","""A""","""reward""","""feedback""","""reward"""
"""AB""",2,"""C2""","""AB""","""C2AB""","""right""","""B""","""reward""","""choice""","""C2AB"""
"""AB""",2,"""C2""","""AB""","""C2AB""","""right""","""B""","""reward""","""feedback""","""reward"""
"""AB""",3,"""C1""","""AB""","""C1BA""","""right""","""A""","""reward""","""choice""","""C1BA"""
…,…,…,…,…,…,…,…,…,…
"""CD""",298,"""C1""","""CD""","""C1CD""","""left""","""C""","""reward""","""feedback""","""reward"""
"""CD""",299,"""C2""","""CD""","""C2DC""","""right""","""C""","""no reward""","""choice""","""C2DC"""
"""CD""",299,"""C2""","""CD""","""C2DC""","""right""","""C""","""no reward""","""feedback""","""no reward"""


In [31]:
import importlib
importlib.reload(world)
nodes = [
    "C1AB",
    "C1BA",
    "C1CD",
    "C1DC",
    "C2AB",
    "C2BA",
    "C2CD",
    "C2DC",
    "reward",
    "no reward",
]
walks = world.walks_mckenzie(design, nodes, 45)
walks

[[[[{'id': 0}],
   tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
            0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
            0., 0., 0., 0., 0., 0., 0., 0., 0.]]),
   [0]],
  [[{'id': 8}],
   tensor([[0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
            0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
            0., 0., 0., 0., 0., 0., 0., 0., 0.]]),
   [0]]],
 [[[{'id': 4}],
   tensor([[0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
            0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
            0., 0., 0., 0., 0., 0., 0., 0., 0.]]),
   [1]],
  [[{'id': 8}],
   tensor([[0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
            0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
            0., 0., 0., 0., 0., 0., 0., 0., 0.]]),
   [0]]],
 [[[{'id': 1}],
   ten